In [1]:
import yfinance as yf
import numpy as np

In [2]:
tickers = ["AMZN", "AAPL"]
ohlcv_data = {}
for ticker in tickers:
    data  = yf.download(ticker, period='1mo', interval='5m')
    data.dropna(how='any', inplace=True)
    ohlcv_data[ticker] = data

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [11]:
def ATR(DF, n=14):
    df = DF.copy()
    df["H-L"] = df["High"] - df["Low"]
    df["H-PC"] = df["High"] - df["Close"].shift(1)
    df["L-PC"] = df["Low"] - df["Close"].shift(1)
    df["TR"] = df[["H-L", "H-PC","L-PC" ]].max(axis=1, skipna=False)
    df["ATR"] = df["TR"].ewm(com=n, min_periods=n).mean()
    return df["ATR"]

def ADX(DF, n=20):
    df = DF.copy()
    df["ATR"]  = ATR(df, n)
    df["upmove"] = df["High"] -  df["High"].shift(1)
    df["downmove"] = df["Low"].shift(1) - df["Low"]
    df["+dm"] = np.where((df["upmove"]>df["downmove"]) & (df["upmove"] > 0), df["upmove"], 0)
    df["-dm"] = np.where((df["downmove"]>df["upmove"]) & (df["downmove"] > 0), df["downmove"], 0)
    df["+di"] = 100 * (df["+dm"]/df["ATR"]).ewm(com = n, min_periods=n).mean()
    df["-di"] = 100 * (df["-dm"]/df["ATR"]).ewm(com = n, min_periods=n).mean()
    df["ADX"] = 100* abs((df["+di"] - df["-di"])/ (df["+di"] + df["-di"])).ewm(span = n, min_periods=n).mean()
    return df["ADX"]


    

In [12]:
for ticker in ohlcv_data:
    ohlcv_data[ticker]["ADX"] = ADX(ohlcv_data[ticker], 20)
print(ohlcv_data["AMZN"]["ADX"])

Datetime
2026-06-15 09:30:00-04:00          NaN
2026-06-15 09:35:00-04:00          NaN
2026-06-15 09:40:00-04:00          NaN
2026-06-15 09:45:00-04:00          NaN
2026-06-15 09:50:00-04:00          NaN
                               ...    
2026-07-14 11:25:00-04:00    29.318055
2026-07-14 11:30:00-04:00    28.289540
2026-07-14 11:35:00-04:00    27.808036
2026-07-14 11:40:00-04:00    27.372388
2026-07-14 11:45:00-04:00    26.978232
Name: ADX, Length: 1510, dtype: float64
